# 04. Binary Sub1bit and Ternary BNN XNOR BiLLM OneBit TWN TTQ

- **Category:** 14_quantisation / 04_vector_lattice_qat_and_lowbit
- **Lineage & Citations:** Courbariaux et al. (2015, *BinaryConnect: Training Deep Neural Networks with binary weights during propagations*); Rastegari et al. (2016, *XNOR-Net: ImageNet Classification Using Binary Convolutional Neural Networks*); Li et al. (2016, *Ternary Weight Networks*); Zhu et al. (2016, *Trained Ternary Quantization*); Xu et al. (2024, *OneBit: Towards Extremely Low-bit Large Language Models*).
- **Purpose:** Eliminate floating-point multiplications by compressing weights to 1-bit ($\pm 1$) or 1.58-bit ($\pm 1, 0$) using optimal mean scaling (XNOR-Net), thresholded ternary states (TWN/TTQ), and rank-1 value-sign decomposition (OneBit).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. QAT STE LSQ LSQ+ PACT DoReFa](03_qat_ste_lsq_lsq_plus_pact_and_dorefa.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. BitNet b1.58 Native Ternary Transformers](05_bitnet_b1_58_native_ternary_transformers.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT apply naive sign binarization without mean/rank-1 scaling $\alpha = \frac{1}{n}\|W\|_1$ (destroys representation scale and causes layer-wise dynamic range collapse); do NOT allow latent weights to drift unboundedly during BNN training without $[-1, 1]$ clipping (latent weight inertia prevents signs from updating).

## Mathematical Foundations & Extreme Low-Bit Quantization

1. **BNN Optimal Scale Derivation (XNOR-Net)**:
   $$\mathbf{B} = \text{sign}(\mathbf{W}), \quad \alpha^* = \arg\min_\alpha \|\mathbf{W} - \alpha \mathbf{B}\|_F^2 = \frac{1}{n}\|\mathbf{W}\|_1$$
2. **Ternary Weight Networks (TWN)**:
   $$T_i = \text{sign}(W_i) \cdot \mathbb{I}[|W_i| > \Delta^*], \quad \Delta^* \approx 0.7 \cdot \frac{1}{n}\|\mathbf{W}\|_1, \quad \alpha^* = \frac{1}{|I_{\Delta}|} \sum_{i \in I_{\Delta}} |W_i|$$
3. **Trained Ternary Quantization (TTQ Asymmetric Scaling)**:
   $$W_i \approx W_p \cdot \mathbb{I}[W_i > \Delta] - W_n \cdot \mathbb{I}[W_i < -\Delta]$$
4. **OneBit Rank-1 Sign-Magnitude Decomposition**:
   $$\mathbf{W} \approx \mathbf{S} \odot (\mathbf{g} \mathbf{h}^\top), \quad \mathbf{S} \in \{-1, +1\}, \mathbf{g} \in \mathbb{R}^{d_{\text{in}} \times 1}_{>0}, \mathbf{h} \in \mathbb{R}^{1 \times d_{\text{out}}}_{>0}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/04_vector_lattice_qat_and_lowbit/04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def bnn_xnor_quantize(w: torch.Tensor):
    signs = torch.where(w >= 0.0, 1.0, -1.0)
    alpha = torch.mean(torch.abs(w), dim=0, keepdim=True)
    return signs * alpha, signs, alpha

def twn_quantize(w: torch.Tensor):
    delta = 0.7 * torch.mean(torch.abs(w), dim=0, keepdim=True)
    t = torch.where(w > delta, 1.0, torch.where(w < -delta, -1.0, 0.0))
    nz = (torch.abs(t) > 0.0).float()
    alpha = torch.sum(torch.abs(w) * nz, dim=0, keepdim=True) / torch.clamp(torch.sum(nz, dim=0, keepdim=True), min=1.0)
    return t * alpha, t, alpha

def ttq_quantize(w: torch.Tensor, delta_factor: float = 0.35):
    delta = delta_factor * torch.max(torch.abs(w), dim=0, keepdim=True).values
    pos = (w > delta).float(); neg = (w < -delta).float()
    w_pos = torch.sum(w * pos, dim=0, keepdim=True) / torch.clamp(torch.sum(pos, dim=0, keepdim=True), min=1.0)
    w_neg = torch.sum(torch.abs(w) * neg, dim=0, keepdim=True) / torch.clamp(torch.sum(neg, dim=0, keepdim=True), min=1.0)
    return pos * w_pos - neg * w_neg, pos - neg, w_pos, w_neg

def onebit_quantize(w: torch.Tensor):
    signs = torch.where(w >= 0.0, 1.0, -1.0)
    abs_w = torch.abs(w)
    g = torch.mean(abs_w, dim=1, keepdim=True)
    h = (torch.sum(abs_w, dim=0, keepdim=True) / torch.clamp(torch.sum(abs_w), min=1e-5)) * float(w.shape[0])
    return signs * (g @ h), signs, g, h

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. BNN and TWN parity
w_bnn, b_s, b_a = bnn_xnor_quantize(weights["w_orig"])
assert float(torch.max(torch.abs(w_bnn - weights["w_bnn"]))) < 1e-5
w_twn, t_t, t_a = twn_quantize(weights["w_orig"])
assert float(torch.max(torch.abs(w_twn - weights["w_twn"]))) < 1e-5

# 3. TTQ and OneBit parity
w_ttq, ttq_t, wp, wn = ttq_quantize(weights["w_orig"], 0.35)
assert float(torch.max(torch.abs(w_ttq - weights["w_ttq"]))) < 1e-5
w_ob, ob_s, ob_g, ob_h = onebit_quantize(weights["w_orig"])
assert float(torch.max(torch.abs(w_ob - weights["w_onebit"]))) < 1e-5
print("All BNN, TWN, TTQ, and OneBit parity assertions verified against golden checkpoint!")

In [ ]:
# Comparative SQNR Benchmark: BNN vs TWN vs TTQ vs OneBit
y_fp32 = weights["out_fp32"]
sig_pwr = float(torch.mean(y_fp32 ** 2))

def eval_sqnr(y_pred: torch.Tensor) -> float:
    mse = float(torch.mean((y_fp32 - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_bnn = eval_sqnr(weights["out_bnn"])
sqnr_twn = eval_sqnr(weights["out_twn"])
sqnr_ttq = eval_sqnr(weights["out_ttq"])
sqnr_onebit = eval_sqnr(weights["out_onebit"])

print(f"BNN (1-bit XNOR) SQNR:   {sqnr_bnn:.2f} dB")
print(f"TWN (1.58-bit Sym) SQNR: {sqnr_twn:.2f} dB (+{sqnr_twn - sqnr_bnn:.2f} dB over BNN)")
print(f"TTQ (1.58-bit Asym) SQNR:{sqnr_ttq:.2f} dB (+{sqnr_ttq - sqnr_bnn:.2f} dB over BNN)")
print(f"OneBit (Rank-1) SQNR:    {sqnr_onebit:.2f} dB")
assert sqnr_twn > sqnr_bnn + 2.0, "Ternary TWN states must yield >2 dB higher SQNR than binary states!"
assert sqnr_ttq > sqnr_bnn + 2.0, "Asymmetric TTQ states must yield >2 dB higher SQNR than binary states!"
assert sqnr_bnn > 3.0, "BNN binarization with mean scaling must maintain viable SQNR (>3 dB)!"

In [ ]:
# Latency Microbenchmark: Fused OneBit Rank-1 Dequantization
d_in, d_out = 4096, 4096
signs_mock = torch.randint(0, 2, (d_in, d_out)).float() * 2.0 - 1.0
g_mock = torch.randn(d_in, 1).abs() * 0.1
h_mock = torch.randn(1, d_out).abs() * 0.1

t0 = time.perf_counter()
for _ in range(50):
    w_deq = signs_mock * (g_mock @ h_mock)
t_onebit = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fused OneBit Rank-1 Dequantization (4096x4096) latency: {t_onebit:.3f} ms")
assert t_onebit < 25.0, "OneBit dequantization must execute in <25 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Optimal Mean Scaling**: Standard binarization requires scaling by $\alpha^* = \frac{1}{n}\|W\|_1$ to preserve weight variance.
2. **Ternary Pruning**: The zero state in $\{-1, 0, +1\}$ prunes $\approx 40\%$ of near-zero weights, providing a $+3\text{--}4\text{ dB}$ SQNR boost at 1.58 bits.
3. **Rank-1 Value-Sign Factorization**: OneBit expresses weights as $S \odot (gh^\top)$, retaining 1-bit footprint while recovering high-rank dynamic range.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.md](../../../markdown/14_quantisation/04_vector_lattice_qat_and_lowbit/04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.md)
- **PyTorch 2.x Module:** [04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.py](../../../pytorch2/14_quantisation/04_vector_lattice_qat_and_lowbit/04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.py)
- **Functional JAX Module:** [04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.py](../../../jax/14_quantisation/04_vector_lattice_qat_and_lowbit/04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.py)
- **Native Rust Candle Track:** [04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.rs](../../../candle/14_quantisation/04_vector_lattice_qat_and_lowbit/04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.rs)
- **Hardware Kernel (CPU / WGSL):** [04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.rs](../../../kernels/14_quantisation/04_vector_lattice_qat_and_lowbit/04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.rs)
- **Golden SafeTensors:** [04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.safetensors](../../../weights/14_quantisation/04_vector_lattice_qat_and_lowbit/04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. QAT STE LSQ LSQ+ PACT DoReFa](03_qat_ste_lsq_lsq_plus_pact_and_dorefa.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 05. BitNet b1.58 Native Ternary Transformers](05_bitnet_b1_58_native_ternary_transformers.ipynb) &rarr;